# PharmGraph — Steps 1–4: Data Cleaning, Graph Construction & Split

This notebook implements Steps 1–4 of the PharmGraph ML pipeline as defined in the team planning document.

| Step | What happens here |
|------|-------------------|
| 1 | Clean and standardize all four edge files |
| 2 | Merge into one unified heterogeneous graph |
| 3 | Set up the link prediction task (gene–drug as target) |
| 4 | Split gene–drug edges into train / val / test (70/15/15) |

**Inputs:** `gene_drug_edges.csv`, `gene_disease_edges.csv`, `variant_drug_edges.csv`, `variant_gene_edges.csv`  
**Outputs:** `unified_graph_edges.csv`, `unified_graph_nodes.csv`, `gene_drug_split.csv`

## 0. Imports & configuration

In [1]:
import pandas as pd
import numpy as np
from sklearn.utils import shuffle
import os

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = "."          # folder containing the four raw CSV files
OUTPUT_DIR = "./outputs"  # folder where processed files will be written
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.15
# TEST_FRAC is the remainder (0.15)

print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)

pandas  : 2.3.1
numpy   : 2.0.2


## 1. Load raw data

In [2]:
gene_drug_raw  = pd.read_csv(os.path.join(DATA_DIR, "gene_drug_edges.csv"))
gene_dis_raw   = pd.read_csv(os.path.join(DATA_DIR, "gene_disease_edges.csv"))
var_drug_raw   = pd.read_csv(os.path.join(DATA_DIR, "variant_drug_edges.csv"))
var_gene_raw   = pd.read_csv(os.path.join(DATA_DIR, "variant_gene_edges.csv"))

print("Raw row counts:")
print(f"  gene_drug:    {len(gene_drug_raw):>7,}")
print(f"  gene_disease: {len(gene_dis_raw):>7,}")
print(f"  variant_drug: {len(var_drug_raw):>7,}")
print(f"  variant_gene: {len(var_gene_raw):>7,}")

Raw row counts:
  gene_drug:    160,480
  gene_disease:   9,425
  variant_drug:   4,785
  variant_gene:   2,830


### 1a. Quick schema inspection

In [3]:
for name, df in [("gene_drug", gene_drug_raw), ("gene_disease", gene_dis_raw),
                 ("variant_drug", var_drug_raw), ("variant_gene", var_gene_raw)]:
    print(f"=== {name} ===")
    print(df.dtypes.to_string())
    print()

=== gene_drug ===
source                object
source_type           object
target                object
target_type           object
source_database       object
source_version        object
interaction_type      object
interaction_score    float64
gene_family           object

=== gene_disease ===
gene_id           object
gene_symbol       object
disease_id        object
disease_name      object
evidence_type     object
association       object
pk_related       float64
pd_related        object
pmids             object
gene_family       object

=== variant_drug ===
source                     object
source_type                object
gene_symbol                object
target                     object
target_type                object
pharmgkb_annotation_id      int64
level_of_evidence          object
evidence_score            float64
phenotype_category         object
phenotypes                 object
pmid_count                  int64
evidence_count              int64
specialty_populatio

In [4]:
# Preview each file
print("gene_drug sample:");    display(gene_drug_raw.head(3))
print("gene_disease sample:"); display(gene_dis_raw.head(3))
print("variant_drug sample:"); display(var_drug_raw.head(3))
print("variant_gene sample:"); display(var_gene_raw.head(3))

gene_drug sample:


,source,source_type,target,target_type,source_database,source_version,interaction_type,interaction_score,gene_family
0,CYP2D6,gene,raclopride,drug,DTC,9/2/20,NaN,0.017709,Cytochrome P450 family 2
1,PPARG,gene,chembl:chembl1833984,drug,DTC,9/2/20,NaN,0.840123,Peroxisome proliferator activated receptors
2,ATAD5,gene,chembl:chembl91609,drug,DTC,9/2/20,NaN,0.177992,AAA ATPases


gene_disease sample:


,gene_id,gene_symbol,disease_id,disease_name,evidence_type,association,pk_related,pd_related,pmids,gene_family
0,PA31744,NQO1,PA151958383,Gastrointestinal Stromal Tumors,"ClinicalAnnotation,VariantAnnotation",associated,NaN,PD,30237583,Flavoproteins
1,PA31744,NQO1,PA164925725,Toxic liver disease,VariantAnnotation,ambiguous,NaN,NaN,17400324;40744310,Flavoproteins
2,PA31744,NQO1,PA166122058,Mucositis,VariantAnnotation,associated,NaN,NaN,33675324,Flavoproteins


variant_drug sample:


,source,source_type,gene_symbol,target,target_type,pharmgkb_annotation_id,level_of_evidence,evidence_score,phenotype_category,phenotypes,pmid_count,evidence_count,specialty_population,last_updated
0,rs951439,variant,RGS4,risperidone,drug,655384602,3,1.75,Efficacy,Schizophrenia,1,1,NaN,2021-03-24
1,rs951439,variant,RGS4,antipsychotics,drug,655384607,3,5.00,Efficacy,Schizophrenia,1,3,NaN,2021-03-24
2,rs4149015,variant,SLCO1B1,pravastatin,drug,655384626,3,0.00,Metabolism/PK,NaN,2,2,Pediatric,2021-03-24


variant_gene sample:


,source,source_type,target,target_type,variant_id,variant_type,rsid,haplotype,hgnc_name,gene_family,primary_gene_family,gene_family_id,gene_family_match_source,in_clinical_variants_tsv,in_interactions_tsv,in_relationships_tsv
0,rs951439,variant,RGS4,gene,183,SNP,rs951439,NaN,regulator of G protein signaling 4,Regulators of G-protein signaling,Regulators of G-protein signaling,720,symbol,True,True,True
1,rs4149015,variant,SLCO1B1,gene,184,SNP,rs4149015,NaN,solute carrier organic anion transporter famil...,Solute carrier organic anion transporter family,Solute carrier organic anion transporter family,2216,symbol,True,True,True
2,rs58597806,variant,UGT1A9,gene,185,SNP,rs58597806,NaN,UDP glucuronosyltransferase family 1 member A9,UDP glucuronosyltransferases,UDP glucuronosyltransferases,363,symbol,True,True,True


## 2. Step 1 — Clean and standardize each edge file

For each file we:
1. Normalize string IDs (lowercase + strip whitespace)
2. Drop rows with missing endpoints
3. Filter to the expected node-type pair
4. Collapse duplicate source–target pairs

In [5]:
def normalize_str(series: pd.Series) -> pd.Series:
    """Lowercase and strip leading/trailing whitespace."""
    return series.astype(str).str.strip().str.lower()


def clean_edges(
    df: pd.DataFrame,
    src_col: str,
    tgt_col: str,
    expected_src_type: str,
    expected_tgt_type: str,
    src_type_col: str = "source_type",
    tgt_type_col: str = "target_type",
) -> pd.DataFrame:
    """
    Generic cleaner:
    - Normalizes src_col and tgt_col
    - Drops rows with null endpoints
    - Keeps only rows matching the expected node types
    - Deduplicates on (src_col, tgt_col)
    """
    df = df.copy()
    df[src_col] = normalize_str(df[src_col])
    df[tgt_col] = normalize_str(df[tgt_col])

    before_null = len(df)
    df = df.dropna(subset=[src_col, tgt_col])
    dropped_null = before_null - len(df)

    if src_type_col in df.columns:
        df = df[df[src_type_col].str.lower() == expected_src_type]
    if tgt_type_col in df.columns:
        df = df[df[tgt_type_col].str.lower() == expected_tgt_type]

    before_dedup = len(df)
    df = df.drop_duplicates(subset=[src_col, tgt_col])
    dropped_dedup = before_dedup - len(df)

    print(f"  null rows dropped : {dropped_null}")
    print(f"  duplicates dropped: {dropped_dedup}")
    print(f"  final rows        : {len(df):,}")
    return df

In [6]:
print("── gene_drug ──────────────────────────")
gd_clean = clean_edges(
    gene_drug_raw,
    src_col="source", tgt_col="target",
    expected_src_type="gene", expected_tgt_type="drug"
)

── gene_drug ──────────────────────────
  null rows dropped : 0
  duplicates dropped: 91474
  final rows        : 69,006


In [7]:
# gene_disease uses different column names — rename first
print("── gene_disease ───────────────────────")
gdis = gene_dis_raw.copy()
gdis["gene_symbol"] = normalize_str(gdis["gene_symbol"])
gdis["disease_id"]  = normalize_str(gdis["disease_id"])
before_null = len(gdis)
gdis = gdis.dropna(subset=["gene_symbol", "disease_id"])
before_dedup = len(gdis)
gdis = gdis.drop_duplicates(subset=["gene_symbol", "disease_id"])
print(f"  null rows dropped : {before_null - before_dedup}")
print(f"  duplicates dropped: {before_dedup - len(gdis)}")
print(f"  final rows        : {len(gdis):,}")

# Rename to unified source/target schema
gdis_clean = gdis.rename(columns={"gene_symbol": "source", "disease_id": "target"})
gdis_clean["source_type"] = "gene"
gdis_clean["target_type"] = "disease"

── gene_disease ───────────────────────
  null rows dropped : 0
  duplicates dropped: 0
  final rows        : 9,425


In [8]:
print("── variant_drug ───────────────────────")
vd_clean = clean_edges(
    var_drug_raw,
    src_col="source", tgt_col="target",
    expected_src_type="variant", expected_tgt_type="drug"
)

── variant_drug ───────────────────────
  null rows dropped : 0
  duplicates dropped: 453
  final rows        : 4,332


In [9]:
print("── variant_gene ───────────────────────")
vg_clean = clean_edges(
    var_gene_raw,
    src_col="source", tgt_col="target",
    expected_src_type="variant", expected_tgt_type="gene"
)

── variant_gene ───────────────────────
  null rows dropped : 0
  duplicates dropped: 0
  final rows        : 2,830


## 3. Step 2 — Build the unified heterogeneous graph

All four relations are merged into one edge table.  
Node IDs are type-prefixed (`gene:cyp2d6`, `drug:warfarin`, etc.) to avoid collisions across node types.

In [10]:
def make_edge_table(
    df: pd.DataFrame,
    relation_type: str,
    src_type: str,
    tgt_type: str,
    src_col: str = "source",
    tgt_col: str = "target",
) -> pd.DataFrame:
    """
    Produces a standardized edge DataFrame with:
      relation_type, source_id, target_id, source_raw, target_raw,
      source_type, target_type
    source_id / target_id are type-prefixed for global uniqueness.
    """
    return pd.DataFrame({
        "relation_type": relation_type,
        "source_id":     src_type + ":" + df[src_col].values,
        "target_id":     tgt_type + ":" + df[tgt_col].values,
        "source_raw":    df[src_col].values,
        "target_raw":    df[tgt_col].values,
        "source_type":   src_type,
        "target_type":   tgt_type,
    })

In [11]:
e_gene_drug    = make_edge_table(gd_clean,   "gene_drug",    "gene",    "drug")
e_gene_disease = make_edge_table(gdis_clean, "gene_disease", "gene",    "disease")
e_var_drug     = make_edge_table(vd_clean,   "variant_drug", "variant", "drug")
e_var_gene     = make_edge_table(vg_clean,   "variant_gene", "variant", "gene")

unified_edges = pd.concat(
    [e_gene_drug, e_gene_disease, e_var_drug, e_var_gene],
    ignore_index=True
)

# Final global dedup (belt-and-suspenders)
before = len(unified_edges)
unified_edges = unified_edges.drop_duplicates(
    subset=["relation_type", "source_id", "target_id"]
)
print(f"Global dedup removed {before - len(unified_edges)} rows")

print("\nEdges per relation type:")
print(unified_edges["relation_type"].value_counts().to_string())
print(f"\nTotal edges: {len(unified_edges):,}")

Global dedup removed 0 rows

Edges per relation type:
relation_type
gene_drug       69006
gene_disease     9425
variant_drug     4332
variant_gene     2830

Total edges: 85,593


In [12]:
# Build node manifest
nodes_src = unified_edges[["source_id", "source_type"]].rename(
    columns={"source_id": "node_id", "source_type": "node_type"}
)
nodes_tgt = unified_edges[["target_id", "target_type"]].rename(
    columns={"target_id": "node_id", "target_type": "node_type"}
)
unified_nodes = (
    pd.concat([nodes_src, nodes_tgt], ignore_index=True)
    .drop_duplicates(subset=["node_id"])
    .sort_values("node_type")
    .reset_index(drop=True)
)

print("Nodes per type:")
print(unified_nodes["node_type"].value_counts().to_string())
print(f"\nTotal unique nodes: {len(unified_nodes):,}")

Nodes per type:
node_type
drug       18481
gene        5292
variant     2826
disease      756

Total unique nodes: 27,355


## 4. Step 3 — Task setup

- **Prediction target:** `gene_drug` edges only  
- **Context edges:** `gene_disease`, `variant_drug`, `variant_gene` (stay in the graph, never predicted)

In [13]:
# Separate the target relation from context edges for clarity
target_edges  = unified_edges[unified_edges["relation_type"] == "gene_drug"].copy()
context_edges = unified_edges[unified_edges["relation_type"] != "gene_drug"].copy()

print(f"Target edges  (gene_drug)  : {len(target_edges):,}")
print(f"Context edges (other 3)    : {len(context_edges):,}")

display(target_edges.head(3))

Target edges  (gene_drug)  : 69,006
Context edges (other 3)    : 16,587


,relation_type,source_id,target_id,source_raw,target_raw,source_type,target_type
0,gene_drug,gene:cyp2d6,drug:raclopride,cyp2d6,raclopride,gene,drug
1,gene_drug,gene:pparg,drug:chembl:chembl1833984,pparg,chembl:chembl1833984,gene,drug
2,gene_drug,gene:atad5,drug:chembl:chembl91609,atad5,chembl:chembl91609,gene,drug


## 5. Step 4 — Train / val / test split (70 / 15 / 15)

- Split applied **only** to gene–drug edges  
- Context edges remain unsplit (all available during training)  
- Only training gene–drug edges should be visible to the model during training to avoid leakage

In [14]:
gd_shuffled = shuffle(target_edges, random_state=RANDOM_SEED).reset_index(drop=True)

n       = len(gd_shuffled)
n_train = int(TRAIN_FRAC * n)
n_val   = int(VAL_FRAC   * n)
# n_test  = n - n_train - n_val  (remainder)

gd_shuffled["split"] = "test"
gd_shuffled.loc[:n_train - 1,              "split"] = "train"
gd_shuffled.loc[n_train:n_train + n_val - 1, "split"] = "val"

print("gene_drug split breakdown:")
counts = gd_shuffled["split"].value_counts()
for split in ["train", "val", "test"]:
    pct = counts[split] / n * 100
    print(f"  {split:<6}: {counts[split]:>6,}  ({pct:.1f}%)")

gene_drug split breakdown:
  train : 48,304  (70.0%)
  val   : 10,350  (15.0%)
  test  : 10,352  (15.0%)


In [15]:
# Propagate split labels back into the main unified edge table
split_map = gd_shuffled.set_index(["source_id", "target_id"])["split"]

unified_edges["split"] = unified_edges.apply(
    lambda row: split_map.get((row["source_id"], row["target_id"]), np.nan)
    if row["relation_type"] == "gene_drug" else np.nan,
    axis=1,
)

print("Split column populated in unified_edges:")
print(unified_edges["split"].value_counts(dropna=False).to_string())

Split column populated in unified_edges:
split
train    48304
NaN      16587
test     10352
val      10350


## 6. Save outputs

In [16]:
# 1. Unified edge table (all 4 relations, split labels for gene_drug)
unified_edges_path = os.path.join(OUTPUT_DIR, "unified_graph_edges.csv")
unified_edges.to_csv(unified_edges_path, index=False)
print(f"Saved: {unified_edges_path}  ({len(unified_edges):,} rows)")

# 2. Node manifest
nodes_path = os.path.join(OUTPUT_DIR, "unified_graph_nodes.csv")
unified_nodes.to_csv(nodes_path, index=False)
print(f"Saved: {nodes_path}  ({len(unified_nodes):,} rows)")

# 3. Gene-drug split table (ML target relation only)
gd_split_out = gd_shuffled[["source_id", "source_raw", "target_id", "target_raw", "split"]].copy()
gd_split_path = os.path.join(OUTPUT_DIR, "gene_drug_split.csv")
gd_split_out.to_csv(gd_split_path, index=False)
print(f"Saved: {gd_split_path}  ({len(gd_split_out):,} rows)")

Saved: ./outputs/unified_graph_edges.csv  (85,593 rows)
Saved: ./outputs/unified_graph_nodes.csv  (27,355 rows)
Saved: ./outputs/gene_drug_split.csv  (69,006 rows)


## 7. Summary

A final overview of everything produced by this notebook.

In [17]:
print("=" * 55)
print("PharmGraph preprocessing — Steps 1–4 complete")
print("=" * 55)

print("\nGraph statistics:")
print(f"  Total edges   : {len(unified_edges):,}")
print(f"  Total nodes   : {len(unified_nodes):,}")
for nt, cnt in unified_nodes["node_type"].value_counts().items():
    print(f"    {nt:<10}: {cnt:,}")

print("\nEdges by relation type:")
for rt, cnt in unified_edges["relation_type"].value_counts().items():
    print(f"  {rt:<20}: {cnt:,}")

print("\nTrain/val/test split (gene_drug only):")
for split in ["train", "val", "test"]:
    cnt = (gd_shuffled["split"] == split).sum()
    pct = cnt / len(gd_shuffled) * 100
    print(f"  {split:<6}: {cnt:,}  ({pct:.1f}%)")

print("\nOutput files:")
for path in [unified_edges_path, nodes_path, gd_split_path]:
    print(f"  {path}")

PharmGraph preprocessing — Steps 1–4 complete

Graph statistics:
  Total edges   : 85,593
  Total nodes   : 27,355
    drug      : 18,481
    gene      : 5,292
    variant   : 2,826
    disease   : 756

Edges by relation type:
  gene_drug           : 69,006
  gene_disease        : 9,425
  variant_drug        : 4,332
  variant_gene        : 2,830

Train/val/test split (gene_drug only):
  train : 48,304  (70.0%)
  val   : 10,350  (15.0%)
  test  : 10,352  (15.0%)

Output files:
  ./outputs/unified_graph_edges.csv
  ./outputs/unified_graph_nodes.csv
  ./outputs/gene_drug_split.csv
